In [ ]:
# WhatsApp Chat Sentiment Analyzer using Google Dataset + LSTM
# ------------------------------------------------------------

# Install libraries
# pip install pandas numpy scikit-learn tensorflow nltk kagglehub

import pandas as pd
import numpy as np
import re
import sys
import subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Install missing packages if needed
for pkg_name, import_name in (("nltk", "nltk"), ("kagglehub", "kagglehub"), ("tensorflow", "tensorflow"), ("scikit-learn", "sklearn")):
    try:
        __import__(import_name)
    except Exception:
        pip_install(pkg_name)

import nltk
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ------------------------------------------------------------
# STEP 1 : DOWNLOAD DATASET FROM KAGGLE
# ------------------------------------------------------------

# Dataset:
# https://www.kaggle.com/datasets/kazanova/sentiment140

path = kagglehub.dataset_download("kazanova/sentiment140")

print("Dataset downloaded to:", path)

# ------------------------------------------------------------
# STEP 2 : LOAD DATASET
# ------------------------------------------------------------

# sentiment140 dataset columns
columns = ["sentiment", "id", "date", "query", "user", "text"]

df = pd.read_csv(
    path + "/training.1600000.processed.noemoticon.csv",
    encoding='latin-1',
    names=columns
)

# Keep only needed columns
df = df[['sentiment', 'text']]

# Reduce dataset size for faster training
df = df.sample(20000, random_state=42)

print(df.head())

# ------------------------------------------------------------
# STEP 3 : CONVERT LABELS
# ------------------------------------------------------------

# Original:
# 0 = negative
# 4 = positive

df['sentiment'] = df['sentiment'].replace(4, 1)

# ------------------------------------------------------------
# STEP 4 : TEXT CLEANING
# ------------------------------------------------------------

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove special characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # remove stopwords
    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

# ------------------------------------------------------------
# STEP 5 : TOKENIZATION
# ------------------------------------------------------------

max_words = 5000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])

X = pad_sequences(sequences, maxlen=max_len)

y = df['sentiment']

# ------------------------------------------------------------
# STEP 6 : SPLIT DATA
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------------------------
# STEP 7 : BUILD LSTM MODEL
# ------------------------------------------------------------

model = Sequential()

model.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

model.add(LSTM(64))

model.add(Dropout(0.5))

model.add(Dense(32, activation='relu'))

model.add(Dense(1, activation='sigmoid'))

# ------------------------------------------------------------
# STEP 8 : COMPILE MODEL
# ------------------------------------------------------------

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ------------------------------------------------------------
# STEP 9 : TRAIN MODEL
# ------------------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# ------------------------------------------------------------
# STEP 10 : EVALUATE MODEL
# ------------------------------------------------------------

y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

# ------------------------------------------------------------
# STEP 11 : PREDICT CUSTOM MESSAGE
# ------------------------------------------------------------

def predict_sentiment(message):

    cleaned = clean_text(message)

    seq = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(seq, maxlen=max_len)

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        return "Positive 😊"
    else:
        return "Negative 😠"

# Example

msg = "I am very happy today"

result = predict_sentiment(msg)

print("\nMessage:", msg)

print("Predicted Sentiment:", result)

ModuleNotFoundError: No module named 'nltk'